# Chapter 2: Using NumPy Special Methods

A beginner-friendly workbook packed with detailed explanations, visual examples, and practice exercises:
1. **2.1 NumPy Special Values**: Missing data (`np.nan`), infinite values (`np.inf`), and boolean detection masks (`np.isnan`, `np.isinf`)
2. **2.2 Array Concatenation**: Joining data along axes using `np.concatenate`, `np.vstack`, and `np.hstack`
3. **2.3 Sorting Arrays**: Multi-axis sorting with `np.sort` and index-based table sorting using `np.argsort`

---
---

## 2.1 NumPy Special Values (NaN and Infinity)

### Concept Summary
In real-world data science, datasets are rarely clean. A weather sensor might shut off during a storm, an online survey respondent might skip questions, or a division might divide by zero. NumPy provides special floating-point values to represent these anomalies without crashing your entire script.

---

### Special Values: Deep Dive with Examples

#### 1. What is NaN (`np.nan`)?
- **Meaning:** Stands for **Not a Number**.
- **What it represents:** Missing values, corrupted data entries, or mathematically undefined operations (like taking the logarithm of a negative number).
- **Rule — The "Contagious" Nature of NaN:** Any mathematical operation performed on a `nan` value will always result in `nan`.
- **Example:**
  ```python
  val = np.nan
  print(val + 5)   # Output: nan
  print(val * 10)  # Output: nan
  print(val == val) # Output: False (NaN is never equal to anything, not even itself!)
  ```

#### 2. Detecting NaNs (`np.isnan`)
Because `val == np.nan` always returns `False`, you **cannot** find NaNs using normal `==` equality checks. Instead, use `np.isnan()`.
- **Syntax:** `np.isnan(array)`
- It evaluates every element and returns a boolean mask of `True` and `False` values.
- **Example:**
  ```python
  raw_scores = np.array([95.0, np.nan, 82.0, np.nan])
  mask = np.isnan(raw_scores)
  print(mask)
  # Output: [False  True False  True]

  # Counting how many NaNs exist:
  print(np.sum(mask))  # Output: 2 (because True counts as 1)
  ```

#### 3. Infinity (`np.inf` and `-np.inf`)
- Represents positive or negative infinity.
- Appears when a calculation exceeds the maximum possible floating-point value or divides a non-zero number by zero.
- **Example:**
  ```python
  large_val = np.inf
  print(large_val + 1000)  # Output: inf
  print(large_val * 2)     # Output: inf
  print(-np.inf < 0)       # Output: True
  ```

#### 4. Detecting Infinity (`np.isinf`) 
- **Syntax:** `np.isinf(array)`
- Scans the array and flags both positive infinity (`inf`) and negative infinity (`-np.inf`) as `True`.
- **Example:**
  ```python
  numbers = np.array([10.0, np.inf, -np.inf, 25.0])
  print(np.isinf(numbers))
  # Output: [False  True  True False]
  ```

#### 5. Filtering Clean Data Using Boolean Masks 
To keep only the valid, clean numbers from an array, combine masks using the bitwise NOT operator (`~`):
- **Example:**
  ```python
  data = np.array([10.0, np.nan, 20.0, np.inf, 30.0])
  # Keep values that are NOT nan and NOT inf
  clean_data = data[~np.isnan(data) & ~np.isinf(data)]
  print(clean_data)
  # Output: [10. 20. 30.]
  ```

In [ ]:
# --- Demo: Special Values and Boolean Masks ---
import numpy as np

# 1. Creating NaN and displaying contagious math
a = np.nan
print("Normal NaN value:     ", a)              # Output: nan
print("Math with NaN (a + 4):", a + 4)          # Output: nan

# 2. Invalid math creating NaN
log_test = np.log([-1, 1, 2])
print("\nLog of [-1, 1, 2]:    ", log_test)     # Output: [       nan 0.         0.69314718]
print("Mask for NaNs:          ", np.isnan(log_test))

# 3. Infinity and isinf mask
inf_array = np.array([np.inf, 10.0, -np.inf, 5.0])
print("\nArray with Inf:       ", inf_array)    # Output: [ inf  10. -inf   5.]
print("Mask for Inf:           ", np.isinf(inf_array))

# 4. Filtering out anomalies
valid_only = inf_array[~np.isinf(inf_array)]
print("Cleaned Numbers:        ", valid_only)    # Output: [10.  5.]

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 1: Sensor Anomaly Detection and Cleaning

**Goal:** Inspect a raw temperature stream containing faulty readings, detect both missing values and infinite spikes, and extract only the valid numbers.

**Requirements:**
1. Create a 1D NumPy array named `raw_temps` containing the following values in order:
   - `23.5`, `np.nan`, `25.1`, `np.inf`, `24.0`, `-np.inf`, and `np.nan`.
2. Generate a boolean mask that locates all `NaN` values, and print the total count of missing readings.
3. Generate a boolean mask that locates all infinite values (`np.inf` or `-np.inf`), and print the mask.
4. Using boolean indexing, extract and print a clean array containing only the valid numerical temperatures (excluding all NaNs and infinities).

<details>
<summary>💡 Need a hint?</summary>

- Which function tests specifically for missing Not-a-Number values?
- True evaluates to 1 when passed into `np.sum()`.
- The bitwise NOT operator (`~`) inverts a boolean mask, turning `True` into `False` and `False` into `True`.
- Use the bitwise AND operator (`&`) to combine exclusion masks.
</details>

In [ ]:
# --- Exercise 1: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 2.2 Array Concatenation and Stacking

### Concept Summary
Concatenation means stitching two or more arrays together into a single combined array. In data analytics, you do this constantly when combining monthly sales records or appending new feature columns to a dataset.

---

### Methods for Joining Arrays

#### 1. The Core Function: `np.concatenate((a, b), axis=...)`
- **Syntax Rule:** The arrays to combine must be passed inside an inner **tuple** or list: `(a, b)`.
- **`axis=0` (Vertical Stacking / Adding Rows):**
  - Stitches arrays top-to-bottom.
  - The number of **columns must match**.
- **`axis=1` (Horizontal Stacking / Adding Columns):**
  - Stitches arrays side-by-side.
  - The number of **rows must match**.
- **Shape Compatibility Rule:** The dimensions must be identical along all axes *except* the one you are concatenating on, or NumPy raises a `ValueError`.

**Visualizing 2D Concatenation:**
```text
Array A (2x3):        Array B (2x3):
[[1, 2, 3],           [[ 7,  8,  9],
 [4, 5, 6]]            [10, 11, 12]]

--- axis=0 (Stack Rows Downwards -> Result is 4x3) ---
[[ 1,  2,  3],
 [ 4,  5,  6],
 [ 7,  8,  9],
 [10, 11, 12]]

--- axis=1 (Join Columns Sideways -> Result is 2x6) ---
[[ 1,  2,  3,  7,  8,  9],
 [ 4,  5,  6, 10, 11, 12]]
```

#### 2. The Vertical Shortcut: `np.vstack((a, b))`
- **Meaning:** Stands for **Vertical Stack**.
- Stacks arrays on top of each other along rows.
- **Convenience with 1D arrays:** If you pass two 1D arrays of shape `(3,)`, `np.vstack` automatically stacks them into a 2D matrix of shape `(2, 3)` without requiring any manual reshaping!
- **Example:**
  ```python
  a = np.array([1, 2, 3])
  b = np.array([4, 5, 6])
  print(np.vstack((a, b)))
  # Output:
  # [[1 2 3]
  #  [4 5 6]]
  ```

#### 3. The Horizontal Shortcut: `np.hstack((a, b))`
- **Meaning:** Stands for **Horizontal Stack**.
- Stacks arrays side-by-side along columns.
- For 1D arrays, it glues them end-to-end to create a longer 1D array.
- For 2D matrices, it appends columns to the right.
- **Example:**
  ```python
  a = np.array([1, 2, 3])
  b = np.array([4, 5, 6])
  print(np.hstack((a, b)))
  # Output: [1 2 3 4 5 6]
  ```

In [ ]:
# --- Demo: Concatenating and Stacking ---
import numpy as np

# Sample arrays
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])
c = np.array([[1, 2, 3], [4, 5, 6]])
d = np.array([[7, 8, 9], [10, 11, 12]])

# 1. 1D concatenation
print("1D Concatenate:             ", np.concatenate((a, b)))

# 2. 2D concatenation along different axes
print("\n2D Concatenate (axis=0, rows):\n", np.concatenate((c, d), axis=0))
print("\n2D Concatenate (axis=1, cols):\n", np.concatenate((c, d), axis=1))

# 3. vstack and hstack shortcuts
print("\n1D arrays using vstack (creates 2D):\n", np.vstack((a, b)))
print("\n1D arrays using hstack (stays 1D):   ", np.hstack((a, b)))
print("\n2D matrices using hstack:\n", np.hstack((c, d)))

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 2: Merging Data Tables and Feature Matrices

**Goal:** Combine independent data tables both vertically (adding new sample records) and horizontally (adding new feature measurements).

**Requirements:**
1. Create two 2D matrices:
   - `batch_1`: A 2x3 matrix with numbers `1` through `6`.
   - `batch_2`: A 2x3 matrix with numbers `7` through `12`.
2. Join `batch_1` and `batch_2` vertically using `np.concatenate` so that the result has 4 rows and 3 columns. Print the resulting array and its shape.
3. Join `batch_1` and `batch_2` horizontally using `np.concatenate` so that the result has 2 rows and 6 columns. Print the resulting array and its shape.
4. Create two 1D arrays: `ids = np.array([101, 102])` and `scores = np.array([88, 92])`.
   - Combine them into a single 2D matrix of shape `(2, 2)` using the appropriate stacking shortcut without manually calling `.reshape()`.

<details>
<summary>💡 Need a hint?</summary>

- Remember that `np.concatenate` requires the arrays to be enclosed inside parentheses: `((arr1, arr2))`.
- Which axis corresponds to rows (vertical), and which corresponds to columns (horizontal)?
- Which helper function automatically stacks two 1D arrays as rows in a 2D matrix?
</details>

In [ ]:
# --- Exercise 2: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>

---

## 2.3 Sorting Arrays and Index Sorting

### Concept Summary
Sorting in NumPy is implemented in optimized C code, making it drastically faster than standard Python loops when ordering large matrices.

---

### Sorting Methods: Deep Dive with Examples

#### 1. Conventional Sorting (`np.sort` vs `arr.sort`)
- **`np.sort(arr)`:** Returns a **new sorted copy** of the array, preserving the original array untouched.
- **`arr.sort()`:** Sorts the array **in place**, mutating the original array and returning `None`.

#### 2. Controlling the Sort Axis
In multi-dimensional arrays, you can specify how the sorting happens using the `axis` parameter:
- **`axis=None` (Global / Flattened Sort):**
  - Flattens the array into a 1D sequence first, then sorts every single number from smallest to largest.
- **`axis=0` (Sort Down Columns):**
  - Treats each column independently, ordering numbers vertically from top to bottom.
- **`axis=1` (Sort Across Rows):**
  - Treats each row independently, ordering numbers horizontally from left to right (this is the default behavior).

**Visualizing Axis Sorting on a 3x3 Matrix:**
```text
Original Matrix:
[[9, 3, 5],
 [6, 1, 9],
 [5, 9, 6]]

np.sort(arr, axis=None) -> [1, 3, 5, 5, 6, 6, 9, 9, 9] (All numbers ordered in 1D)

np.sort(arr, axis=0) (Columns sorted downwards):
[[5, 1, 5],   <- Column 0: 9, 6, 5 becomes 5, 6, 9
 [6, 3, 6],   <- Column 1: 3, 1, 9 becomes 1, 3, 9
 [9, 9, 9]]   <- Column 2: 5, 9, 6 becomes 5, 6, 9

np.sort(arr, axis=1) (Rows sorted across left-to-right):
[[3, 5, 9],   <- Row 0: 9, 3, 5 becomes 3, 5, 9
 [1, 6, 9],   <- Row 1: 6, 1, 9 becomes 1, 6, 9
 [5, 6, 9]]   <- Row 2: 5, 9, 6 becomes 5, 6, 9
```

---

#### 3. Sorting by Index with `np.argsort`
Instead of returning the sorted numbers themselves, **`np.argsort()` returns the index positions that would sort the array**.
- **Think of it like a position detective:** It tells you where each sorted value originally came from.
- **Example:**
  ```python
  scores = np.array([50, 10, 40, 20])
  # Values:  50 (idx 0), 10 (idx 1), 40 (idx 2), 20 (idx 3)
  # Smallest to largest: 10, 20, 40, 50
  # Original index order: 1,  3,  2,  0

  order = np.argsort(scores)
  print(order)  # Output: [1 3 2 0]

  # Use the indices to rearrange the original array:
  print(scores[order])  # Output: [10 20 40 50]
  ```

#### 4. Why `np.argsort` is Essential in Data Analytics
In a real data table (like students and their grades), if you use standard `np.sort`, it will sort the grade column independently and completely detach grades from student IDs!
With `np.argsort`, you find the row order based on the grade column, and reorder the **entire table** so rows stay intact:
```python
table = np.array([
    [101, 85],  # Student 101, Score 85
    [102, 60],  # Student 102, Score 60
    [103, 95]   # Student 103, Score 95
])

# 1. Get row index order that sorts by the Score column (column index 1)
sorted_rows = np.argsort(table[:, 1])  # Yields indices: [1, 0, 2]

# 2. Reorder the entire table safely:
print(table[sorted_rows])
# Output:
# [[102  60]
#  [101  85]
#  [103  95]]
```

In [ ]:
# --- Demo: Sorting and argsort ---
import numpy as np

# Sample matrix
arr = np.array([
    [9, 3, 5],
    [6, 1, 9],
    [5, 9, 6]
])

print("Initial Matrix:\n", arr)
print("\nFlattened sort (axis=None):\n", np.sort(arr, axis=None))
print("\nColumn sort (axis=0):\n", np.sort(arr, axis=0))
print("\nRow sort (axis=1):\n", np.sort(arr, axis=1))

# Index sorting with argsort
numbers = np.array([1, 4, 7, 2, 5])
# Sort numbers by their remainder when divided by 3 (custom sorting rule)
remainder_indices = np.argsort(numbers % 3)
print("\nOriginal numbers:                     ", numbers)
print("Indices sorted by remainder (num % 3): ", remainder_indices)
print("Numbers sorted by remainder:           ", numbers[remainder_indices])

<div style="height: 20px; background: linear-gradient(to bottom, red 0%, red 2px, rgba(255, 0, 0, 0.5) 3px, transparent 100%); border-radius: 6px 6px 0 0; margin: 15px 0;"></div>

### Exercise 3: Multi-Axis Sorting and Table Record Alignment

**Goal:** Order matrices across different dimensions and rearrange an entire multi-column table based on values in a single target column.

**Requirements:**
1. Create a 3x3 array containing the following rows:
   - Row 0: `[14,  3,  9]`
   - Row 1: `[ 2, 18,  5]`
   - Row 2: `[11,  1,  8]`
2. Print a completely flattened, sorted 1D array containing all 9 elements.
3. Sort the matrix column-by-column (downwards) without modifying the original array.
4. Given this table of employee records where **Column 0 = Employee ID** and **Column 1 = Performance Score**:
   ```text
   [[201,  78],
    [202,  94],
    [203,  61],
    [204,  85]]
   ```
5. Use index sorting on the performance scores (Column 1) to determine the row order from lowest to highest score.
6. Use those index positions to reorder the entire table so that each Employee ID remains paired with their score.

<details>
<summary>💡 Need a hint?</summary>

- What setting for `axis` instructs `np.sort` to collapse the matrix into a single 1D sorted stream?
- For column-wise sorting, which axis index represents column directions (downwards)?
- To extract a single column for ranking, use slice notation `[:, target_column]`.
- Which function returns the ordered row index positions without altering the numbers directly?
- How do you pass a list of row indices into a 2D array to reorder entire rows at once?
</details>

In [ ]:
# --- Exercise 3: Your Code Here ---

<div style="height: 24px; background: linear-gradient(to top, red 0%, rgba(255, 0, 0, 0.4) 35%, transparent 100%); border-radius: 0 0 8px 8px; margin: 15px 0;"></div>